# Semantic Chunking Experiment

## Objective

The objective of this experiment is to develop and evaluate a semantic chunking strategy for the RAG pipeline using the structured document tree produced from Markdown.

The previous chunking pipeline operated directly on extracted text and relied primarily on `RecursiveCharacterTextSplitter`. The new approach uses the document's structural hierarchy as the primary semantic signal before applying size-based splitting.

---


## Current Document Representation

The document is now represented through the following pipeline:

```text
PDF
 ↓
Markdown Extraction
 ↓
Markdown Tree
 ↓
Semantic Chunking
 ↓
Final Chunks
```

The Markdown tree preserves document hierarchy such as:

* Headings
* Sub-headings
* Paragraphs
* Lists
* Tables
* Nested sections

This structure will be used to identify meaningful semantic units.

---


In [9]:
"""
Step 0
reading the file and preprocess it to have the tree
"""
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))


import pymupdf
import pymupdf4llm
from src.preprocessing.ingestion_preprocessing import HeaderInfoBuilder
FILE_PATH ="../src/pdfs/logical_reasoning.pdf"


header_builder = HeaderInfoBuilder()
hdr_info = header_builder.build(FILE_PATH)


pymupdf4llm.use_layout(False)

doc = pymupdf.open(FILE_PATH)

markdown = pymupdf4llm.to_markdown(
    doc,
    show_progress=True,
    # use_ocr=False,
    hdr_info=hdr_info
)


# Markdown -> Tree
from src.preprocessing.markdown_parser import build_tree

markdown_tree = build_tree(text=markdown)

Processing ../src/pdfs/logical_reasoning.pdf...


100%|██████████| 239/239 [00:20<00:00, 11.47it/s]



# 1. Semantic Chunking Strategy

## Core Idea

The chunker will **not replace `RecursiveCharacterTextSplitter`**.

Instead, the chunking process will use two layers:

```text
Markdown Tree
      ↓
Semantic Unit Identification
      ↓
Semantic Grouping
      ↓
RecursiveCharacterTextSplitter
      ↓
Final Chunks
```

The first layer determines **where semantic boundaries exist**.

The second layer controls **the physical size of the chunks**.

---


# 2. Semantic Units

A semantic unit represents a meaningful section of the document.

For example:

```text
Logical Reasoning
│
└── Verbal Reasoning
    │
    ├── Types of Problems
    │   ├── Number Series
    │   ├── Letter Series
    │   └── Symbol Series
    │
    └── Solving Strategies
```

Potential semantic units are:

```text
Number Series
Letter Series
Symbol Series
Solving Strategies
```

The document hierarchy provides the initial boundaries instead of relying only on character count. 

# 3. Hierarchy as the Primary Semantic Signal

The Markdown tree provides information about the relationship between sections.

For example:

```text
H2: Conditional Probability
│
├── H3: Definition
├── H3: Formula
├── H3: Example
└── H3: Applications
```

These nodes are related because they belong to the same parent topic.

The chunking strategy should preserve this relationship while avoiding unnecessarily large chunks.

---



# 4. Candidate Semantic Units

The first version of the experiment will generate candidate semantic units from the document tree.

Each candidate should contain:

```text
Heading / Context
+
Associated Content
+
Relevant Child Content
```

Example:

```text
Topic: Number Series

Content:
...
```

The candidate units will then be evaluated for size and semantic coherence.

---



# 5. Size Control

Semantic boundaries alone are not sufficient.

A semantic section may be too large to use as a single retrieval chunk.

Therefore:

```text
Semantic Unit
      │
      ├── Within acceptable size
      │        ↓
      │      Keep
      │
      └── Too large
               ↓
      RecursiveCharacterTextSplitter
               ↓
          Multiple chunks
```

The existing `RecursiveCharacterTextSplitter` remains responsible for controlling chunk size.

---

# 6. Context Preservation

When a child section becomes a chunk, its hierarchical context should be preserved.

For example:

```text
Logical Reasoning
→ Verbal Reasoning
→ Number Series
```

should provide sufficient context for the resulting chunk to remain meaningful during retrieval.

The exact representation of this context will be evaluated during the experiment.

---

# 7. Embedding-Based Semantic Boundaries

Embedding similarity is considered a **second-stage enhancement**, not the initial mechanism.

The initial POC will first evaluate whether the Markdown hierarchy alone provides sufficient semantic boundaries.

If structural boundaries are insufficient, embeddings can be introduced:

```text
Semantic Units
      ↓
Generate Embeddings
      ↓
Calculate Similarity
      ↓
Detect Semantic Transitions
      ↓
Refine Boundaries
```

This allows the system to use document structure first and semantic similarity only where required.

---

# 8. Experiment

The experiment will evaluate:

1. How well Markdown hierarchy represents semantic sections.
2. Whether individual tree nodes form useful retrieval units.
3. Whether related child nodes should be grouped.
4. How large semantic units become.
5. Where size-based splitting is required.
6. Whether hierarchical context improves chunk quality.
7. Whether embedding-based boundary detection is actually necessary.

---

# 9. Evaluation

The generated chunks will be inspected for:

### Semantic coherence

Does each chunk discuss a reasonably focused concept?

### Context

Can the chunk be understood without retrieving the entire parent topic?

### Size

Is the chunk small enough for efficient retrieval?

### Boundary quality

Does the chunk avoid cutting concepts at arbitrary character boundaries?

### Retrieval usefulness

Would the chunk be a useful retrieval result for a query about its specific subtopic?

---

# 10. Expected Architecture

```text
PDF
 │
 ▼
Markdown Extraction
 │
 ▼
Markdown Tree
 │
 ▼
Semantic Unit Identification
 │
 ▼
Semantic Grouping
 │
 ▼
Size Evaluation
 │
 ├───────────────┐
 │               │
 ▼               ▼
Small Unit    Large Unit
 │               │
 │               ▼
 │      RecursiveCharacter
 │        Text Splitter
 │               │
 └───────┬───────┘
         ▼
    Final Chunks
         │
         ▼
     Embeddings
         │
         ▼
     Vector Store
```

---

# 11. Experiment Result

## Observations

*To be filled after inspecting the generated chunks.*

## Problems Identified

*To be filled during experimentation.*

## Improvements Required

*To be filled based on observed failures.*

---

# 12. Final Decision

*To be completed after the experiment.*

Possible outcomes:

* Markdown hierarchy is sufficient for semantic boundaries.
* Markdown hierarchy + size-based splitting is sufficient.
* Markdown hierarchy requires embedding-based refinement.
* Additional structural heuristics are required.

The final implementation should be based on the observed behavior of the generated chunks rather than assumptions made before experimentation.
